# Lab 7: VQE Molecular Simulation
## Applied Quantum Computing — Chapter 7

**The Simulation Revolution: Drugs, Materials, and Climate**

This notebook provides two labs:
- **Lab 7 (Regular):** H₂ VQE ground-state calculation and bond-distance sweep
- **Lab 7 (Advanced):** LiH with UCCSD vs. hardware-efficient ansatz comparison

No local installation required — run entirely in Google Colab.

---
**Learning Objectives:**
1. Run a VQE calculation on H₂ and observe ground-state energy convergence
2. Plot energy vs. bond distance to identify equilibrium geometry
3. Compare UCCSD and hardware-efficient ansatz for LiH
4. Benchmark quantum results against classical Hartree-Fock
5. Connect simulation results to pharmaceutical R&D economics

## Setup: Install Required Packages

Run this cell first. Installation takes 2-3 minutes in Colab.

In [ ]:
# Install Qiskit and Qiskit Nature
!pip install qiskit==0.45.0 qiskit-nature==0.7.2 qiskit-algorithms==0.3.0 pyscf matplotlib numpy scipy --quiet
print('Installation complete.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Qiskit imports
from qiskit_algorithms import VQE, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA, SLSQP
from qiskit.primitives import Estimator

# Qiskit Nature imports
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit.circuit.library import EfficientSU2

print('All imports successful.')

---
# PART 1 — Lab 7 Regular: H₂ VQE Ground-State Calculation

The hydrogen molecule (H₂) is the simplest molecule chemistry has — two protons, two electrons. It is also the proving ground of every quantum chemistry algorithm ever invented. If an algorithm can't get H₂ right, it won't get anything right.

In this section we will:
1. Compute the ground-state energy at equilibrium bond distance
2. Sweep bond distance from compressed to stretched
3. Plot the potential energy surface (PES)
4. Identify the equilibrium bond length

## Step 1: H₂ VQE at Equilibrium Bond Distance

In [ ]:
def run_h2_vqe(bond_distance_angstrom=0.735):
    """
    Run VQE for H2 at a specified bond distance.
    Returns ground state energy in Hartree.
    """
    # Define H2 geometry
    driver = PySCFDriver(
        atom=f'H .0 .0 .0; H .0 .0 {bond_distance_angstrom}',
        basis='sto3g',
        charge=0,
        spin=0
    )
    
    # Build the electronic structure problem
    problem = driver.run()
    
    # Map to qubit Hamiltonian using Parity mapper (reduces qubit count by 2 for H2)
    mapper = ParityMapper(num_particles=problem.num_particles)
    qubit_op = mapper.map(problem.second_q_ops()[0])
    
    # Initial state: Hartree-Fock
    hf_state = HartreeFock(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper
    )
    
    # Ansatz: UCCSD
    ansatz = UCCSD(
        problem.num_spatial_orbitals,
        problem.num_particles,
        mapper,
        initial_state=hf_state
    )
    
    # Classical optimizer
    optimizer = SLSQP(maxiter=300)
    
    # Estimator (statevector simulator for noise-free results)
    estimator = Estimator()
    
    # Run VQE
    vqe = VQE(estimator, ansatz, optimizer)
    result = vqe.compute_minimum_eigenvalue(qubit_op)
    
    # Add nuclear repulsion to get total energy
    nuclear_repulsion = problem.nuclear_repulsion_energy
    total_energy = result.eigenvalue.real + nuclear_repulsion
    
    return total_energy

# Run at equilibrium bond distance
print('Running H2 VQE at equilibrium bond distance (0.735 Å)...')
e_equilibrium = run_h2_vqe(0.735)
print(f'Ground-state energy: {e_equilibrium:.6f} Hartree')
print(f'Reference value:    -1.137270 Hartree')
print(f'Error: {abs(e_equilibrium - (-1.137270)):.6f} Hartree')

## Step 2: VQE Convergence Plot

Watch how VQE starts from a poor guess and converges to the ground state.

In [ ]:
# Track energy convergence during optimization
energy_history = []

def callback(count, params, value, meta):
    energy_history.append(value)

driver = PySCFDriver(atom='H .0 .0 .0; H .0 .0 0.735', basis='sto3g')
problem = driver.run()
mapper = ParityMapper(num_particles=problem.num_particles)
qubit_op = mapper.map(problem.second_q_ops()[0])

hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper, initial_state=hf_state)

optimizer = COBYLA(maxiter=200)
estimator = Estimator()
vqe = VQE(estimator, ansatz, optimizer, callback=callback)
result = vqe.compute_minimum_eigenvalue(qubit_op)

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(range(len(energy_history)), energy_history, 'b-', linewidth=1.5, alpha=0.8)
plt.axhline(y=-1.137270 - problem.nuclear_repulsion_energy, color='r', 
            linestyle='--', label='Reference ground state energy')
plt.xlabel('Optimization Iteration', fontsize=12)
plt.ylabel('Electronic Energy (Hartree)', fontsize=12)
plt.title('VQE Convergence: H₂ Ground-State Energy', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('h2_vqe_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nConverged in {len(energy_history)} iterations')
print(f'Final energy: {energy_history[-1]:.6f} Hartree')

## Step 3: Bond Distance Sweep — The Potential Energy Surface

In [ ]:
# Sweep bond distances from 0.3 Å to 3.0 Å
bond_distances = np.arange(0.3, 3.1, 0.1)
vqe_energies = []
hf_energies = []

print('Running bond distance sweep (this may take a few minutes)...')
for d in bond_distances:
    try:
        driver = PySCFDriver(atom=f'H .0 .0 .0; H .0 .0 {d:.2f}', basis='sto3g')
        problem = driver.run()
        mapper = ParityMapper(num_particles=problem.num_particles)
        qubit_op = mapper.map(problem.second_q_ops()[0])
        
        hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
        
        # HF energy (classical baseline)
        numpy_solver = NumPyMinimumEigensolver()
        exact_result = numpy_solver.compute_minimum_eigenvalue(qubit_op)
        exact_energy = exact_result.eigenvalue.real + problem.nuclear_repulsion_energy
        
        # Simple VQE with hardware-efficient ansatz for speed
        ansatz = EfficientSU2(qubit_op.num_qubits, reps=1, entanglement='linear')
        optimizer = COBYLA(maxiter=500)
        estimator = Estimator()
        vqe = VQE(estimator, ansatz, optimizer)
        vqe_result = vqe.compute_minimum_eigenvalue(qubit_op)
        vqe_energy = vqe_result.eigenvalue.real + problem.nuclear_repulsion_energy
        
        vqe_energies.append(vqe_energy)
        hf_energies.append(exact_energy)  # Using exact as reference
        print(f'  d={d:.2f} Å: E={vqe_energy:.4f} Ha')
    except Exception as e:
        vqe_energies.append(np.nan)
        hf_energies.append(np.nan)
        print(f'  d={d:.2f} Å: error - {e}')

print('Sweep complete.')

In [ ]:
# Plot the potential energy surface
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(bond_distances, vqe_energies, 'bo-', linewidth=2, markersize=5, label='VQE (Hardware-Efficient Ansatz)')
ax.plot(bond_distances, hf_energies, 'r--', linewidth=1.5, alpha=0.7, label='Exact (FCI Reference)')

# Mark equilibrium
valid = [(d, e) for d, e in zip(bond_distances, vqe_energies) if not np.isnan(e)]
if valid:
    min_idx = np.argmin([e for _, e in valid])
    eq_dist, eq_energy = valid[min_idx]
    ax.axvline(x=eq_dist, color='g', linestyle=':', alpha=0.7, label=f'VQE Equilibrium ≈ {eq_dist:.2f} Å')
    ax.scatter([eq_dist], [eq_energy], color='g', s=150, zorder=5)

ax.axvline(x=0.735, color='orange', linestyle=':', alpha=0.7, label='Experimental Equilibrium (0.735 Å)')

ax.set_xlabel('Bond Distance (Angström)', fontsize=13)
ax.set_ylabel('Total Energy (Hartree)', fontsize=13)
ax.set_title('H₂ Potential Energy Surface via VQE', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([-1.2, 0.2])

# Annotation
ax.annotate('Equilibrium: minimum\n(equilibrium bond length)', 
            xy=(0.735, -1.13), xytext=(1.2, -0.8),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10)

plt.tight_layout()
plt.savefig('h2_potential_energy_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nVQE equilibrium bond length: {eq_dist:.3f} Å (experimental: 0.735 Å)')

## Step 4: Deliverable — Memo Template

Copy this template and complete it for your submission.

---
### MEMO TEMPLATE

**TO:** Drug Discovery Leadership  
**FROM:** [Your Name]  
**RE:** Why the H₂ Energy Curve Matters for Drug Discovery  
**DATE:** [Date]

---

**What the Energy Curve Shows**

[Explain in 2-3 sentences: what does the minimum of the potential energy curve represent? What happens at very short bond distances (left side of curve)? What happens as bond distance grows large (right side)?]

**Connection to Drug Binding Affinity**

[Explain in 2-3 sentences: how does ground-state energy minimization relate to how a drug molecule binds to its protein target? What property does the simulation need to predict accurately for a drug to work as designed?]

**NPV Impact**

[Calculation: assume improving binding affinity prediction accuracy by 15% reduces Phase II failure rate by 10% (from 50% to 40%) for a drug with $500M/year peak revenue and 10% discount rate. Calculate the NPV impact of this improvement. Show your work.]

**Hardware Limitation**

[Identify one specific limitation — circuit depth, qubit count, error rate, or ansatz expressibility — that prevents scaling this H₂ calculation to a 50-atom drug-like molecule on today's NISQ hardware.]

---
*Word count target: 350-400 words*

---
# PART 2 — Lab 7 Advanced: LiH and the Limits of Today's Hardware

Lithium hydride (LiH) is a three-electron system (after frozen-core approximation) complex enough to illustrate the challenge ahead. We will compare two ansatz choices: the physics-motivated UCCSD and the hardware-efficient alternative.

## Step 1: Set Up LiH Problem

In [ ]:
# LiH at equilibrium bond distance (1.596 Å)
lih_driver = PySCFDriver(
    atom='Li .0 .0 .0; H .0 .0 1.596',
    basis='sto3g',
    charge=0,
    spin=0
)

lih_problem = lih_driver.run()

# Reduce active space for tractability on NISQ hardware
# Keep only 2 electrons in 3 orbitals (core 1s of Li frozen)
transformer = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=3)
lih_problem_reduced = transformer.transform(lih_problem)

mapper = JordanWignerMapper()
lih_qubit_op = mapper.map(lih_problem_reduced.second_q_ops()[0])

print(f'LiH molecular orbitals (full): {lih_problem.num_spatial_orbitals}')
print(f'Active space orbitals: {lih_problem_reduced.num_spatial_orbitals}')
print(f'Qubits required (JW): {lih_qubit_op.num_qubits}')
print(f'Number of Hamiltonian terms: {len(lih_qubit_op)}')

## Step 2: Classical Hartree-Fock Baseline

In [ ]:
# Exact diagonalization (FCI reference)
numpy_solver = NumPyMinimumEigensolver()
fci_result = numpy_solver.compute_minimum_eigenvalue(lih_qubit_op)
fci_energy = fci_result.eigenvalue.real + lih_problem_reduced.nuclear_repulsion_energy
print(f'FCI (exact) energy: {fci_energy:.6f} Hartree')

# Hartree-Fock energy (classical baseline — no correlation)
hf_state = HartreeFock(
    lih_problem_reduced.num_spatial_orbitals,
    lih_problem_reduced.num_particles,
    mapper
)
# HF energy computed by running VQE with frozen (non-parameterized) HF ansatz
from qiskit.circuit import QuantumCircuit
hf_circuit = hf_state
estimator = Estimator()
hf_expectation = estimator.run([hf_circuit], [lih_qubit_op]).result().values[0]
hf_energy = hf_expectation.real + lih_problem_reduced.nuclear_repulsion_energy
print(f'Hartree-Fock energy:   {hf_energy:.6f} Hartree')
print(f'Correlation energy (FCI - HF): {fci_energy - hf_energy:.6f} Hartree')
print('\n(Correlation energy is what classical HF misses; VQE can recover it)')

## Step 3: UCCSD Ansatz VQE

In [ ]:
# UCCSD ansatz — physically motivated
uccsd_ansatz = UCCSD(
    lih_problem_reduced.num_spatial_orbitals,
    lih_problem_reduced.num_particles,
    mapper,
    initial_state=hf_state
)

print(f'UCCSD ansatz:')
print(f'  Parameters: {uccsd_ansatz.num_parameters}')
print(f'  Circuit depth: {uccsd_ansatz.decompose().depth()}')

uccsd_energy_history = []
def uccsd_callback(count, params, value, meta):
    uccsd_energy_history.append(value)

optimizer = SLSQP(maxiter=500)
estimator = Estimator()
vqe_uccsd = VQE(estimator, uccsd_ansatz, optimizer, callback=uccsd_callback)
uccsd_result = vqe_uccsd.compute_minimum_eigenvalue(lih_qubit_op)
uccsd_energy = uccsd_result.eigenvalue.real + lih_problem_reduced.nuclear_repulsion_energy

print(f'\nUCCSD VQE energy: {uccsd_energy:.6f} Hartree')
print(f'Error vs FCI: {abs(uccsd_energy - fci_energy):.6f} Hartree')
print(f'Iterations: {len(uccsd_energy_history)}')

## Step 4: Hardware-Efficient Ansatz VQE

In [ ]:
# Hardware-efficient ansatz — fewer layers, more noise-tolerant
n_qubits = lih_qubit_op.num_qubits
he_ansatz = EfficientSU2(n_qubits, reps=3, entanglement='full')

print(f'Hardware-Efficient ansatz:')
print(f'  Parameters: {he_ansatz.num_parameters}')
print(f'  Circuit depth: {he_ansatz.decompose().depth()}')

he_energy_history = []
def he_callback(count, params, value, meta):
    he_energy_history.append(value)

# Start from random initial point
import numpy as np
np.random.seed(42)
initial_point = np.random.uniform(-np.pi, np.pi, he_ansatz.num_parameters)

optimizer = COBYLA(maxiter=1000)
estimator = Estimator()
vqe_he = VQE(estimator, he_ansatz, optimizer, initial_point=initial_point, callback=he_callback)
he_result = vqe_he.compute_minimum_eigenvalue(lih_qubit_op)
he_energy = he_result.eigenvalue.real + lih_problem_reduced.nuclear_repulsion_energy

print(f'\nHardware-Efficient VQE energy: {he_energy:.6f} Hartree')
print(f'Error vs FCI: {abs(he_energy - fci_energy):.6f} Hartree')
print(f'Iterations: {len(he_energy_history)}')

## Step 5: Comparison Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Convergence comparison
ax1.plot(range(len(uccsd_energy_history)), 
         [e + lih_problem_reduced.nuclear_repulsion_energy for e in uccsd_energy_history],
         'b-', linewidth=1.5, label='UCCSD (physics-motivated)', alpha=0.8)
ax1.plot(range(len(he_energy_history)),
         [e + lih_problem_reduced.nuclear_repulsion_energy for e in he_energy_history],
         'orange', linewidth=1.5, label='Hardware-Efficient (EfficientSU2)', alpha=0.8)
ax1.axhline(y=fci_energy, color='r', linestyle='--', linewidth=2, label=f'FCI Reference ({fci_energy:.4f} Ha)')
ax1.axhline(y=hf_energy, color='gray', linestyle=':', linewidth=1.5, label=f'Hartree-Fock ({hf_energy:.4f} Ha)')
ax1.set_xlabel('Optimization Iteration', fontsize=12)
ax1.set_ylabel('Total Energy (Hartree)', fontsize=12)
ax1.set_title('LiH VQE Convergence: UCCSD vs Hardware-Efficient', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Energy accuracy bar chart
methods = ['HF\n(Classical)', 'HW-Efficient\n(VQE)', 'UCCSD\n(VQE)', 'FCI\n(Exact)']
energies = [hf_energy, he_energy, uccsd_energy, fci_energy]
colors = ['gray', 'orange', 'blue', 'red']

bars = ax2.bar(methods, energies, color=colors, alpha=0.75, edgecolor='black')
ax2.set_ylabel('Total Energy (Hartree)', fontsize=12)
ax2.set_title('LiH Ground-State Energy: Method Comparison', fontsize=12, fontweight='bold')

# Add error labels
for bar, energy, method in zip(bars, energies, methods):
    error = abs(energy - fci_energy)
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.002,
             f'Δ={error:.4f}', ha='center', va='bottom', fontsize=9)

ax2.set_ylim([min(energies) - 0.05, max(energies) + 0.03])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('lih_vqe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== SUMMARY ===')
print(f'{'Method':<25} {'Energy (Ha)':<15} {'Error vs FCI (Ha)':<20}')
print('-' * 60)
for method, energy in [('Hartree-Fock', hf_energy), 
                        ('Hardware-Efficient VQE', he_energy),
                        ('UCCSD VQE', uccsd_energy),
                        ('FCI (Exact)', fci_energy)]:
    print(f'{method:<25} {energy:<15.6f} {abs(energy-fci_energy):<20.6f}')

## Step 6: H₂O — Where It Breaks (Optional)

Attempt the same workflow for water. Document what breaks.

In [ ]:
# H2O — this is where hardware limits become visible
try:
    h2o_driver = PySCFDriver(
        atom='O .0 .0 .0; H .0 0.757 0.586; H .0 -0.757 0.586',
        basis='sto3g'
    )
    h2o_problem = h2o_driver.run()
    
    print(f'H2O full problem:')
    print(f'  Spatial orbitals: {h2o_problem.num_spatial_orbitals}')
    print(f'  Electrons: {sum(h2o_problem.num_particles)}')
    
    # Reduce active space aggressively
    transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4)
    h2o_reduced = transformer.transform(h2o_problem)
    
    mapper_h2o = JordanWignerMapper()
    h2o_qubit_op = mapper_h2o.map(h2o_reduced.second_q_ops()[0])
    
    print(f'\nActive space H2O:')
    print(f'  Active orbitals: {h2o_reduced.num_spatial_orbitals}')
    print(f'  Qubits required (JW): {h2o_qubit_op.num_qubits}')
    print(f'  Hamiltonian terms: {len(h2o_qubit_op)}')
    print(f'\nFull H2O (no active space reduction):')
    full_mapper = JordanWignerMapper()
    full_op = full_mapper.map(h2o_problem.second_q_ops()[0])
    print(f'  Qubits required: {full_op.num_qubits}')
    print(f'  Hamiltonian terms: {len(full_op)}')
    print(f'  Statevector RAM required: ~{2**full_op.num_qubits * 16 / 1e9:.1f} GB')
    print('\n→ This is where a laptop runs out of memory for classical simulation of quantum circuits.')
    print('  Real quantum hardware would not have this memory constraint — the wavefunction is the hardware.')
    
except Exception as e:
    print(f'Error: {e}')
    print('This error itself illustrates the hardware limitation!')

## Advanced Deliverable — Analysis Template

Write a 600-word analysis answering: **What hardware improvement would be required to extend VQE to a 50-atom drug-like molecule?**

Your analysis should address:
1. **Qubit count:** How many qubits would a 50-atom drug-like molecule require under JW mapping? (Research typical active-space sizes for drug fragments.)
2. **Circuit depth:** UCCSD circuit depth scales as O(N⁴) with system size. What depth would be required?
3. **Error rate requirement:** What error rate would be needed to maintain chemical accuracy (1 kcal/mol ≈ 0.0016 Hartree) for the required circuit depth?
4. **Gap to today:** Today's best hardware achieves ~0.1-0.5% gate error rates. What improvement factor is needed?
5. **Timeline estimate:** Given current improvement rates in qubit quality and error correction overhead, when might this threshold be reachable?

**Include your LiH convergence plots as figures in your writeup.**

---
## Notebook Complete

**Files generated:**
- `h2_vqe_convergence.png` — VQE convergence for H₂
- `h2_potential_energy_surface.png` — Bond distance sweep
- `lih_vqe_comparison.png` — UCCSD vs hardware-efficient comparison

**Next steps:**
- Complete the Memo (Lab 7 Regular deliverable)
- Complete the 600-word analysis (Lab 7 Advanced deliverable)
- Post to the Chapter 7 discussion: pick one molecular bottleneck and estimate its R&D acceleration NPV

---
*Applied Quantum Computing — Chapter 7 | Dr. Ernesto Lee*